# Lezione 47 — Una reward function da zero

> **Modulo:** Feedback e preference training · **Tempo stimato:** 30 minuti
> **Prerequisiti:** Lezione 46 (coppie), Lezione 9 (loss).
>
> Obiettivo pratico unico: addestrare in NumPy un **reward model** che dia a
> ogni memoria un punteggio, tale che chosen > rejected, con la loss di
> Bradley-Terry.

## Teoria essenziale

Un **reward model** $r_\phi(x)$ assegna un punteggio scalare a un elemento. Non
conosciamo il "vero" punteggio: conosciamo solo **preferenze** (Lezione 46). Il
modello di **Bradley-Terry** lega le due cose:

$$P(\text{chosen} \succ \text{rejected}) = \sigma\big(r_\phi(x_w) - r_\phi(x_l)\big)$$

Si addestra $r_\phi$ massimizzando la probabilita' delle preferenze osservate,
cioe' minimizzando $-\log\sigma(r_w - r_l)$ su tutte le coppie. E' la base del
reward model di RLHF (Christiano et al., 2017; Ouyang et al., 2022).

### Un esperimento controllato

Per verificare che il *meccanismo* funzioni, usiamo un setup controllato: fissiamo
noi una **reward vera** $w^\star$ (che di solito non conosceremmo) su feature
sintetiche, generiamo le preferenze da quella reward + rumore, e vediamo se il
reward model la **recupera** partendo solo dalle coppie. Su feedback reale
sostituiresti le feature sintetiche con quelle delle memorie — la matematica non
cambia. *(La Lezione 47 evita di fingere che feature deboli come "lunghezza del
testo" predicano l'importanza: lo mostreremmo come un fallimento, non come un
successo.)*

In [1]:
import numpy as np

rng = np.random.default_rng(47)

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

d = 4                                   # dimensione delle feature
w_star = np.array([1.5, -1.0, 0.5, 0.8])   # la reward VERA (di solito ignota)

# genero coppie: due elementi casuali; 'chosen' e' quello con reward vera piu'
# alta, ma con rumore (a volte l'etichetta e' sbagliata, come col feedback umano)
n = 200
Xa = rng.normal(size=(n, d))
Xb = rng.normal(size=(n, d))
diff_vero = (Xa - Xb) @ w_star
prob_a_meglio = sigmoid(diff_vero)                 # Bradley-Terry con la reward vera
a_scelto = rng.random(n) < prob_a_meglio           # etichetta rumorosa
Xw = np.where(a_scelto[:, None], Xa, Xb)           # feature del chosen
Xl = np.where(a_scelto[:, None], Xb, Xa)           # feature del rejected
print(f"coppie: {n}, dim feature: {d}")
print("reward vera w*:", w_star)

coppie: 200, dim feature: 4
reward vera w*: [ 1.5 -1.   0.5  0.8]


In [2]:
# addestro il reward lineare r(x)=w·x con la loss di Bradley-Terry
w = rng.normal(size=d) * 0.01     # pesi quasi nulli: si parte ~a caso
lr = 0.3

def accuratezza(w):
    return float((Xw @ w > Xl @ w).mean())

print(f"accuratezza iniziale: {accuratezza(w):.2f}  (~0.5 = caso)")
for passo in range(400):
    p = sigmoid(Xw @ w - Xl @ w)                         # P(chosen > rejected)
    grad = -((1 - p)[:, None] * (Xw - Xl)).mean(axis=0)  # d(-log sigma)/dw
    w -= lr * grad
    if passo % 100 == 0 or passo == 399:
        L = -np.log(sigmoid(Xw @ w - Xl @ w) + 1e-12).mean()
        print(f"passo {passo:3d}: loss {L:.4f}  accuratezza {accuratezza(w):.2f}")

accuratezza iniziale: 0.34  (~0.5 = caso)
passo   0: loss 0.6363  accuratezza 0.81
passo 100: loss 0.3683  accuratezza 0.83
passo 200: loss 0.3676  accuratezza 0.84
passo 300: loss 0.3676  accuratezza 0.84
passo 399: loss 0.3676  accuratezza 0.84


Leggi la salita: da accuratezza ~0.5 (caso) il reward impara a ordinare le
coppie. E soprattutto la direzione dei pesi appresi si allinea alla reward vera
$w^\star$ — pur non avendola mai vista, solo attraverso le preferenze.

In [3]:
# coseno tra reward appresa e reward vera: ~1 se ha recuperato la direzione
cos = float(w @ w_star / (np.linalg.norm(w) * np.linalg.norm(w_star)))
print("reward appresa:", np.round(w, 3))
print("reward vera   :", w_star)
print(f"allineamento (coseno) con la reward vera: {cos:.3f}")

reward appresa: [ 1.581 -1.05   0.318  0.781]
reward vera   : [ 1.5 -1.   0.5  0.8]
allineamento (coseno) con la reward vera: 0.995


## Il progetto: Memory AI Lab, passo 47

In [4]:
def allena_reward(Xw, Xl, passi=400, lr=0.3):
    w = np.zeros(Xw.shape[1])
    for _ in range(passi):
        p = sigmoid(Xw @ w - Xl @ w)
        w -= lr * (-((1 - p)[:, None] * (Xw - Xl)).mean(axis=0))
    return w

w2 = allena_reward(Xw, Xl)
acc = float((Xw @ w2 > Xl @ w2).mean())
cos2 = float(w2 @ w_star / (np.linalg.norm(w2) * np.linalg.norm(w_star)))
# controlli di non-regressione: ordina meglio del caso E recupera la direzione vera
assert acc > 0.6, acc
assert cos2 > 0.9, cos2
print(f"reward addestrato: accuratezza {acc:.2f}, allineamento con la vera {cos2:.3f}")

reward addestrato: accuratezza 0.84, allineamento con la vera 0.995


## Riepilogo (max 8 punti)

1. Un **reward model** $r_\phi(x)$ da' un punteggio scalare a un elemento.
2. Non conosciamo il vero punteggio, solo **preferenze**.
3. **Bradley-Terry**: $P(w\succ l)=\sigma(r_w-r_l)$.
4. Si addestra minimizzando $-\log\sigma(r_w-r_l)$ sulle coppie.
5. Da pesi nulli (accuratezza ~0.5) il reward impara a ordinare.
6. I pesi dicono quali **feature** contano.
7. E' la base del reward model di RLHF.
8. Il reward addestrato e' riusabile su qualunque memoria.

## Quiz

1. Perche' non si puo' addestrare il reward con una regressione sul "vero voto"?
2. Cosa rappresenta $\sigma(r_w - r_l)$?
3. Cosa significa accuratezza 0.5 all'inizio?

*(Risposte: 1. il vero voto assoluto non esiste/non e' osservato: abbiamo solo
confronti; 2. la probabilita' modellata che chosen sia preferita a rejected;
3. il reward ordina a caso, come lanciare una moneta.)*